In [1]:
import tensorflow as tf
import tensorflow.keras.models as load_models
import pickle
import numpy as np
import pandas as pd

In [7]:
## Load the trained model
model = load_models.load_model('ann_model.h5')

## Load the scaler
with open('scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

## load the label encoder gender
with open('label_encoder_gender.pkl', 'rb') as f:
    label_encoder_gender = pickle.load(f)

## load the one hot encoder geography
with open('onehot_encoder_geo.pkl', 'rb') as f:
    one_hot_encoder_geography = pickle.load(f)


In [8]:
# Example input data
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

In [9]:
## one hot encode the geography
geography_encoded = one_hot_encoder_geography.transform([[input_data['Geography']]]).toarray()
geography_encoded_df = pd.DataFrame(geography_encoded, columns=one_hot_encoder_geography.get_feature_names_out(['Geography']))
geography_encoded_df

d:\Generative AI\ANN Classification\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [10]:
input_data_encoded = pd.DataFrame([input_data])
input_data_encoded

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [11]:
input_data_encoded['Gender'] = label_encoder_gender.transform(input_data_encoded['Gender'])
input_data_encoded

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


In [12]:
## Concatation one hot encoded data with the original data
input_data_final = pd.concat([input_data_encoded.drop('Geography', axis=1), geography_encoded_df], axis=1)
input_data_final

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [13]:
## Scaling the input data
input_data_scaled = scaler.transform(input_data_final)
input_data_scaled
## Predicting the output

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [15]:
## Predicting the output
prediction = model.predict(input_data_scaled)
prediction

1/1 [==============================] - 0s 32ms/step


array([[0.02923021]], dtype=float32)

In [19]:
# Prediction probality
prediction_prob = prediction[0][0]
if prediction_prob > 0.5:
    print(f"The customer is likely to churn with a probability of {prediction_prob:.2f}")
else:
    print(f"The customer is not likely to churn with a probability of {1 - prediction_prob:.2f}")

The customer is not likely to churn with a probability of 0.97
